In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'GLD'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data

In [ ]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

print(f"Computing indicators for {TICKER}...")
close_f = close.astype(float)
high_f = high.astype(float)
low_f = low.astype(float)

# ATR (core indicator for this strategy)
atr_14 = talib.ATR(high_f, low_f, close_f, timeperiod=14)

# Moving Averages
sma_20 = talib.SMA(close_f, timeperiod=20)
sma_50 = talib.SMA(close_f, timeperiod=50)
ema_12 = talib.EMA(close_f, timeperiod=12)
ema_26 = talib.EMA(close_f, timeperiod=26)

# MACD
macd, macd_signal, macd_hist = talib.MACD(close_f, fastperiod=12, slowperiod=26, signalperiod=9)

# RSI
rsi_14 = talib.RSI(close_f, timeperiod=14)

# Stochastic RSI
stoch_k, stoch_d = talib.STOCHRSI(close_f, timeperiod=14, fastk_period=5, fastd_period=3, fastd_matype=0)

# Build indicators dataframe
indicators_df = pd.DataFrame({
    'ATR_14': atr_14,
    'SMA_20': sma_20, 'SMA_50': sma_50,
    'EMA_12': ema_12, 'EMA_26': ema_26,
    'MACD': macd, 'MACD_Signal': macd_signal, 'MACD_Hist': macd_hist,
    'RSI_14': rsi_14,
    'StochRSI_K': stoch_k, 'StochRSI_D': stoch_d
}, index=stock_data.index)

print(f"\nIndicators computed. Shape: {indicators_df.shape}")
print("\nLast 5 rows:")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise ValueError("No Close column found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

def select_series(df, ticker, col_name):
    if isinstance(df.columns, pd.MultiIndex):
        if (col_name, ticker) in df.columns:
            s = df[(col_name, ticker)]
        else:
            cols = [c for c in df.columns if col_name in str(c)]
            if not cols:
                raise ValueError(f"No {col_name} column found")
            s = df[cols[0]]
    else:
        s = df[col_name]
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'
high_s = select_series(stock_data, TICKER, 'High')
high_s.name = 'high'
low_s = select_series(stock_data, TICKER, 'Low')
low_s.name = 'low'

# Simple split
TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()
train_high  = high_s.iloc[:split_idx].copy()
val_high    = high_s.iloc[split_idx:].copy()
train_low   = low_s.iloc[:split_idx].copy()
val_low     = low_s.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} \u2192 {train_close.index[-1].date()} | val={val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")
print(f"Train: {len(train_close)} bars | Val: {len(val_close)} bars")
print(f"Split ratio: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}")

ATR VOLATILITY BREAKOUT GRID SEARCH - TRAINING SET
---------------------------------------------------

**Concept:** When price breaks beyond N * ATR from a moving average, it signals a real trend. ATR measures "normal" volatility, so breakouts beyond it are statistically significant.

**Signal Logic:**
- **Entry:** Close > SMA + (ATR_mult * ATR) — price breaks above upper ATR band
- **Exit:** Close < SMA - (ATR_mult * ATR) — price breaks below lower ATR band
- All signals shifted by 1 bar to avoid lookahead bias

**Parameters:** `atr_period`, `sma_period`, `atr_mult`

In [ ]:
# Define Parameter Ranges for ATR Volatility Breakout

from itertools import product

atr_period_range = [7, 10, 14, 20, 25, 30]
sma_period_range = [10, 20, 30, 40, 50]
atr_mult_range   = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

print("ATR Volatility Breakout Parameter Ranges:")
print(f"  ATR Period:    {atr_period_range}")
print(f"  SMA Period:    {sma_period_range}")
print(f"  ATR Multiplier: {atr_mult_range}")

# Generate all combinations
atr_combinations = list(product(atr_period_range, sma_period_range, atr_mult_range))

print(f"\nTotal combinations: {len(atr_combinations)}")
print("\nFirst 10 combinations preview:")
for i, (ap, sp, am) in enumerate(atr_combinations[:10]):
    print(f"  {i+1}. ATR_period={ap}, SMA_period={sp}, ATR_mult={am}")

In [ ]:
# Initialize ATR Volatility Breakout Results Collection System

grid_search_results = []

print("ATR Volatility Breakout Results Collection System Initialized")
print(f"   - Will test {len(atr_combinations)} ATR parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

metrics_to_collect = [
    "atr_period", "sma_period", "atr_mult",
    "total_return", "annualized_return",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio", "omega_ratio",
    "information_ratio", "tail_ratio", "deflated_sharpe_ratio",
    "max_drawdown", "volatility", "ulcer_index",
    "win_rate", "total_trades", "avg_trade_duration", "expectancy",
    "profit_factor", "sqn",
    "payoff_ratio", "largest_win", "largest_loss",
    "avg_win_amount", "avg_loss_amount", "winning_streak", "losing_streak",
    "recovery_factor", "gain_to_pain_ratio", "serenity_index"
]

print("Metrics to collect for each ATR combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the ATR Volatility Breakout grid search!")

In [ ]:
# ATR VOLATILITY BREAKOUT GRID SEARCH ON TRAINING DATA

print("INITIATING ATR VOLATILITY BREAKOUT GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: ATR Volatility Breakout (price vs SMA \u00b1 N*ATR)")
print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% per trade (fees + slippage)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(atr_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing sequentially with progress every 100 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_close_vals = train_close.values.astype(float)
train_idx = train_close.index
years = max((train_idx[-1] - train_idx[0]).days / 365.25, 1e-9)

# Need high and low for ATR
if isinstance(stock_data.columns, pd.MultiIndex):
    train_high_vals = stock_data[('High', TICKER)].astype(float).iloc[:split_idx].values
    train_low_vals = stock_data[('Low', TICKER)].astype(float).iloc[:split_idx].values
else:
    train_high_vals = stock_data['High'].astype(float).iloc[:split_idx].values
    train_low_vals = stock_data['Low'].astype(float).iloc[:split_idx].values

for combo_num, (atr_period, sma_period, atr_mult) in enumerate(atr_combinations, 1):
    try:
        atr = talib.ATR(train_high_vals, train_low_vals, train_close_vals, timeperiod=atr_period)
        sma = talib.SMA(train_close_vals, timeperiod=sma_period)
        upper_band = sma + (atr_mult * atr)
        lower_band = sma - (atr_mult * atr)

        upper_s = pd.Series(upper_band, index=train_idx).shift(1)
        lower_s = pd.Series(lower_band, index=train_idx).shift(1)
        close_s = pd.Series(train_close_vals, index=train_idx)

        entries_raw = (close_s > upper_s)  # price breaks above upper ATR band
        exits_raw = (close_s < lower_s)    # price breaks below lower ATR band

        # 1-bar execution delay
        entries = entries_raw.shift(1).fillna(False).astype(bool)
        exits = exits_raw.shift(1).fillna(False).astype(bool)

        pf = vbt.Portfolio.from_signals(close=train_close, entries=entries, exits=exits,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        trades = pf.trades
        total_trades = len(trades)

        if total_trades < 10:
            skipped_low_trades += 1
            continue

        trades_per_year = total_trades / years
        total_return = float(pf.total_return())
        annualized_return = float(pf.annualized_return(freq='D'))
        max_drawdown = float(pf.max_drawdown())
        volatility = float(pf.annualized_volatility(freq='D'))
        sharpe_ratio = float(pf.sharpe_ratio(freq='D'))
        sortino_ratio = float(pf.sortino_ratio(freq='D'))
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan

        win_rate_pct = np.nan; profit_factor = np.nan; expectancy = 0.0
        avg_win_amount = 0.0; avg_loss_amount = 0.0; largest_win = 0.0; largest_loss = 0.0
        winning_streak = np.nan; losing_streak = np.nan; avg_trade_duration = np.nan; sqn = np.nan

        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]; neg = tr[tr < 0]
            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            profit_factor = gains / losses if losses > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
            largest_win = float(pos.max()) if len(pos) else 0.0
            largest_loss = float(neg.min()) if len(neg) else 0.0
            sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            try:
                winning_streak = int(trades.winning_streak())
                losing_streak = int(trades.losing_streak())
            except: pass

        returns = pf.returns()
        cum = (1 + returns).cumprod(); peak = cum.cummax(); dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount > 0 else np.inf
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan

        grid_search_results.append({
            "atr_period": atr_period, "sma_period": sma_period, "atr_mult": atr_mult,
            "total_return": total_return, "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio, "sortino_ratio": sortino_ratio, "calmar_ratio": calmar_ratio,
            "omega_ratio": omega_ratio, "information_ratio": np.nan, "tail_ratio": np.nan,
            "deflated_sharpe_ratio": np.nan, "max_drawdown": max_drawdown, "volatility": volatility,
            "ulcer_index": ulcer_index, "win_rate": win_rate_pct, "total_trades": total_trades,
            "avg_trade_duration": avg_trade_duration, "expectancy": expectancy,
            "profit_factor": profit_factor, "sqn": sqn, "payoff_ratio": payoff_ratio,
            "largest_win": largest_win, "largest_loss": largest_loss,
            "avg_win_amount": avg_win_amount, "avg_loss_amount": avg_loss_amount,
            "winning_streak": winning_streak, "losing_streak": losing_streak,
            "recovery_factor": recovery_factor, "gain_to_pain_ratio": gain_to_pain,
            "serenity_index": serenity_index, "trades_per_year": trades_per_year
        })
        successful_tests += 1
    except Exception as e:
        failed_tests += 1

    if combo_num % 100 == 0:
        progress_pct = (combo_num / total_combinations) * 100
        print(f"\U0001f504 Progress: {combo_num}/{total_combinations} ({progress_pct:.1f}%) | \u2714 {successful_tests} successful | Skipped: {skipped_low_trades}")

print("\n" + "=" * 70)
print("ATR VOLATILITY BREAKOUT GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Skipped (< 10 trades): {skipped_low_trades}")
print(f"Failed: {failed_tests}")
print(f"Success rate: {(successful_tests/total_combinations)*100:.1f}%")
print(f"\n\u2714 Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    print("\n" + "=" * 70)
    print("\U0001f3c6 TOP 10 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    top_10 = results_df_preview.nlargest(10, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_10.iterrows(), 1):
        print(f"\n#{rank} - ATR(period={int(row['atr_period'])}, sma={int(row['sma_period'])}, mult={row['atr_mult']:.1f})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    print("\n" + "=" * 70)

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

FREQ = "1D"
results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("\U0001f3c6 TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")
    print("=" * 90)

    top_5 = results_df.nlargest(5, 'sharpe_ratio')
    oos_results = []
    val_vals = val_close.values.astype(float)
    val_idx = val_close.index
    val_high_vals = val_high.values.astype(float)
    val_low_vals = val_low.values.astype(float)

    for _, strategy in top_5.iterrows():
        ap = int(strategy['atr_period']); sp = int(strategy['sma_period']); am = float(strategy['atr_mult'])
        atr = talib.ATR(val_high_vals, val_low_vals, val_vals, timeperiod=ap)
        sma = talib.SMA(val_vals, timeperiod=sp)
        upper_band = sma + (am * atr)
        lower_band = sma - (am * atr)
        upper_s_val = pd.Series(upper_band, index=val_idx).shift(1)
        lower_s_val = pd.Series(lower_band, index=val_idx).shift(1)
        close_s_val = pd.Series(val_vals, index=val_idx)
        e_raw = (close_s_val > upper_s_val)
        x_raw = (close_s_val < lower_s_val)
        e = e_raw.shift(1).fillna(False).astype(bool)
        x = x_raw.shift(1).fillna(False).astype(bool)
        pf_val = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x,
                                            init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ)
        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ))
        oos_ret = float(pf_val.total_return())
        oos_dd = float(pf_val.max_drawdown())
        trades = pf_val.trades; oos_trades = len(trades)
        oos_wr = np.nan; oos_pf = np.nan
        if oos_trades > 0:
            tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
            if tr.size > 0:
                p = tr[tr > 0]; n = tr[tr < 0]
                oos_wr = (len(p)/len(tr))*100 if len(tr) > 0 else np.nan
                oos_pf = p.sum()/abs(n.sum()) if len(n) > 0 and abs(n.sum()) > 0 else np.inf
        oos_results.append({
            'Rank': len(oos_results)+1, 'ATR_Period': ap, 'SMA_Period': sp, 'ATR_Mult': am,
            'IS_Sharpe': float(strategy['sharpe_ratio']), 'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']), 'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe, 'OOS_Return': oos_ret, 'OOS_MaxDD': oos_dd,
            'OOS_WinRate': oos_wr, 'OOS_Trades': oos_trades, 'OOS_ProfitFactor': oos_pf,
            'Sharpe_Degradation': ((oos_sharpe - strategy['sharpe_ratio'])/abs(strategy['sharpe_ratio'])*100) if strategy['sharpe_ratio'] != 0 else np.nan,
            'Return_Degradation': ((oos_ret - strategy['total_return'])/abs(strategy['total_return'])*100) if strategy['total_return'] != 0 else np.nan
        })

    oos_df = pd.DataFrame(oos_results).sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)
    oos_df['OOS_Rank'] = range(1, len(oos_df)+1)
    print("\n\U0001f4ca COMPREHENSIVE COMPARISON TABLE (Sorted by OOS Sharpe)")
    print("=" * 90)
    display_df = pd.DataFrame({
        'IS\u2192OOS Rank': oos_df['Rank'].astype(str)+'\u2192'+oos_df['OOS_Rank'].astype(str),
        'Strategy': oos_df.apply(lambda x: f"ATR({int(x['ATR_Period'])},{int(x['SMA_Period'])},{x['ATR_Mult']:.1f})", axis=1),
        'IS Sharpe': oos_df['IS_Sharpe'].map('{:.3f}'.format),
        'OOS Sharpe': oos_df['OOS_Sharpe'].map('{:.3f}'.format),
        'Sharpe \u0394%': oos_df['Sharpe_Degradation'].map('{:+.1f}%'.format),
        'IS Return': oos_df['IS_Return'].map('{:.1%}'.format),
        'OOS Return': oos_df['OOS_Return'].map('{:.1%}'.format),
        'Return \u0394%': oos_df['Return_Degradation'].map('{:+.1f}%'.format),
        'OOS Trades': oos_df['OOS_Trades'].astype(int),
        'OOS WinRate': oos_df['OOS_WinRate'].map('{:.1f}%'.format),
        'OOS PF': oos_df['OOS_ProfitFactor'].map('{:.2f}'.format)
    })
    print(display_df.to_string(index=False))
    bo = oos_df.iloc[0]
    print("\n"+"="*90)
    print(f"\u2728 BEST OUT-OF-SAMPLE PERFORMER")
    print("="*90)
    print(f"Strategy: ATR(period={int(bo['ATR_Period'])}, sma={int(bo['SMA_Period'])}, mult={bo['ATR_Mult']:.1f})")
    print(f"In-Sample Rank:        #{int(bo['Rank'])}")
    print(f"Out-of-Sample Rank:    #1")
    print(f"OOS Sharpe Ratio:      {bo['OOS_Sharpe']:.3f}")
    print(f"OOS Return:            {bo['OOS_Return']:.2%}")
    print(f"OOS Max Drawdown:      {bo['OOS_MaxDD']:.2%}")
    print(f"OOS Win Rate:          {bo['OOS_WinRate']:.1f}%")
    print(f"OOS Profit Factor:     {bo['OOS_ProfitFactor']:.2f}")
    print(f"OOS Total Trades:      {int(bo['OOS_Trades'])}")
    print(f"Sharpe Degradation:    {bo['Sharpe_Degradation']:+.1f}%")
    print("="*90)

    # Equity curves
    bi = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_ap, b_sp, b_am = int(bi['atr_period']), int(bi['sma_period']), float(bi['atr_mult'])

    def _atr_sig(ps, hs, ls, ap, sp, am):
        v = ps.values.astype(float); idx = ps.index
        hv = hs.values.astype(float); lv = ls.values.astype(float)
        atr = talib.ATR(hv, lv, v, timeperiod=ap)
        sma = talib.SMA(v, timeperiod=sp)
        ub = pd.Series(sma + (am * atr), index=idx).shift(1)
        lb = pd.Series(sma - (am * atr), index=idx).shift(1)
        cs = pd.Series(v, index=idx)
        er = (cs > ub); xr = (cs < lb)
        e = er.shift(1).fillna(False).astype(bool)
        x = xr.shift(1).fillna(False).astype(bool)
        return e, x

    et, xt = _atr_sig(train_close, train_high, train_low, b_ap, b_sp, b_am)
    pft = vbt.Portfolio.from_signals(close=train_close, entries=et, exits=xt, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    ev, xv = _atr_sig(val_close, val_high, val_low, b_ap, b_sp, b_am)
    pfv = vbt.Portfolio.from_signals(close=val_close, entries=ev, exits=xv, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(train_close.index, (1+pft.returns()).cumprod().values, label='Train (IS)', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(val_close.index, (1+pfv.returns()).cumprod().values, label='Validation (OOS)', color='orange', linewidth=1.5, alpha=0.8)
    ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')
    ax.set_title(f'ATR(period={b_ap}, sma={b_sp}, mult={b_am:.1f}) - Equity Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12); ax.set_ylabel('Cumulative Returns', fontsize=12)
    ax.grid(True, alpha=0.3); ax.legend(loc='best'); plt.tight_layout(); plt.show()

PARAMETER SENSITIVITY ANALYSIS
------------------------------

For each parameter (`atr_period`, `sma_period`, `atr_mult`), we vary it while holding the other two fixed at their best values.

This reveals which parameters the strategy is most sensitive to, and whether the optimal region is robust or fragile.

In [ ]:
# PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    atr_base = int(best['atr_period']); sma_base = int(best['sma_period']); mult_base = float(best['atr_mult'])
    base_is_sharpe = float(best['sharpe_ratio']); base_oos_sharpe = np.nan
    try:
        e_c, x_c = _atr_sig(val_close, val_high, val_low, atr_base, sma_base, mult_base)
        pf_c = vbt.Portfolio.from_signals(close=val_close, entries=e_c, exits=x_c, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        base_oos_sharpe = float(pf_c.sharpe_ratio(freq='D'))
    except: pass
    print(f"\U0001f52c SENSITIVITY ANALYSIS - Base: ATR(period={atr_base}, sma={sma_base}, mult={mult_base:.1f})")
    print(f"IS Sharpe: {base_is_sharpe:.3f}" + (f" | OOS Sharpe: {base_oos_sharpe:.3f}" if not np.isnan(base_oos_sharpe) else ""))

    atr_sens  = sorted(set([5, 7, 10, 14, 20, 25, 30, 35, 40]))
    sma_sens  = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60]))
    mult_sens = sorted(set([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]))

    combos_a = [(a, sma_base, mult_base) for a in atr_sens]
    combos_s = [(atr_base, s, mult_base) for s in sma_sens]
    combos_m = [(atr_base, sma_base, m) for m in mult_sens]
    all_combos = combos_a + combos_s + combos_m

    def eval_both(ap, sp, am):
        e_is, x_is = _atr_sig(train_close, train_high, train_low, ap, sp, am)
        pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        e_oos, x_oos = _atr_sig(val_close, val_high, val_low, ap, sp, am)
        pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        return {'atr_period': ap, 'sma_period': sp, 'atr_mult': am,
                'is_sharpe': float(pf_is.sharpe_ratio(freq='D')), 'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D'))}

    print(f"\n\U0001f504 Testing {len(all_combos)} parameter variations...")
    rows = []
    for c in all_combos:
        try: rows.append(eval_both(*c))
        except: pass

    if not rows:
        print("\u274c No sensitivity results.")
    else:
        sens_df = pd.DataFrame(rows)
        av = sens_df[(sens_df['sma_period']==sma_base)&(sens_df['atr_mult']==mult_base)].copy().sort_values('atr_period')
        sv = sens_df[(sens_df['atr_period']==atr_base)&(sens_df['atr_mult']==mult_base)].copy().sort_values('sma_period')
        mv = sens_df[(sens_df['atr_period']==atr_base)&(sens_df['sma_period']==sma_base)].copy().sort_values('atr_mult')
        av['is_sharpe_delta'] = (av['is_sharpe']-base_is_sharpe)/abs(base_is_sharpe)*100
        sv['is_sharpe_delta'] = (sv['is_sharpe']-base_is_sharpe)/abs(base_is_sharpe)*100
        mv['is_sharpe_delta'] = (mv['is_sharpe']-base_is_sharpe)/abs(base_is_sharpe)*100
        if not np.isnan(base_oos_sharpe):
            av['oos_sharpe_delta'] = (av['oos_sharpe']-base_oos_sharpe)/abs(base_oos_sharpe)*100
            sv['oos_sharpe_delta'] = (sv['oos_sharpe']-base_oos_sharpe)/abs(base_oos_sharpe)*100
            mv['oos_sharpe_delta'] = (mv['oos_sharpe']-base_oos_sharpe)/abs(base_oos_sharpe)*100

        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        fig.suptitle(f'Sensitivity - Base: ATR(period={atr_base}, sma={sma_base}, mult={mult_base:.1f})', fontsize=16, fontweight='bold')
        for ci, (pn, var, pc, bv) in enumerate([('ATR Period', av, 'atr_period', atr_base), ('SMA Period', sv, 'sma_period', sma_base), ('ATR Multiplier', mv, 'atr_mult', mult_base)]):
            colors = ['red' if x<-10 else 'orange' if x<0 else 'lightgreen' if x<10 else 'darkgreen' for x in var['is_sharpe_delta']]
            axes[0,ci].bar(var[pc].astype(str), var['is_sharpe_delta'], color=colors, edgecolor='black', linewidth=0.5)
            axes[0,ci].axhline(0, color='black', linewidth=1.5)
            bv_str = str(bv)
            x_labels = list(var[pc].astype(str))
            if bv_str in x_labels:
                axes[0,ci].axvline(x_labels.index(bv_str), color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={bv}')
            axes[0,ci].set_xlabel(pn, fontsize=11, fontweight='bold'); axes[0,ci].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[0,ci].set_title(f'{pn} - In-Sample', fontsize=12, fontweight='bold')
            axes[0,ci].grid(axis='y', alpha=0.3); axes[0,ci].legend(fontsize=10)
            if not np.isnan(base_oos_sharpe):
                colors2 = ['red' if x<-10 else 'orange' if x<0 else 'lightgreen' if x<10 else 'darkgreen' for x in var['oos_sharpe_delta']]
                axes[1,ci].bar(var[pc].astype(str), var['oos_sharpe_delta'], color=colors2, edgecolor='black', linewidth=0.5)
                axes[1,ci].axhline(0, color='black', linewidth=1.5)
                if bv_str in x_labels:
                    axes[1,ci].axvline(x_labels.index(bv_str), color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={bv}')
                axes[1,ci].set_xlabel(pn, fontsize=11, fontweight='bold'); axes[1,ci].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
                axes[1,ci].set_title(f'{pn} - Out-of-Sample', fontsize=12, fontweight='bold')
                axes[1,ci].grid(axis='y', alpha=0.3); axes[1,ci].legend(fontsize=10)
        plt.tight_layout(); plt.show()

        print("\n\U0001f4cb SENSITIVITY SUMMARY"); print("="*80)
        summary_data = []
        for pn, var, pc in [('ATR Period', av, 'atr_period'), ('SMA Period', sv, 'sma_period'), ('ATR Multiplier', mv, 'atr_mult')]:
            summary_data.append({
                'Parameter': pn,
                'IS Range': f"{var['is_sharpe'].min():.3f} - {var['is_sharpe'].max():.3f}",
                'IS Max \u0394%': f"{var['is_sharpe_delta'].min():.1f}%",
                'OOS Range': f"{var['oos_sharpe'].min():.3f} - {var['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
                'OOS Max \u0394%': f"{var['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in var else 'N/A',
                'Sensitivity': '\u26a0\ufe0f HIGH' if abs(var['is_sharpe_delta'].min()) > 40 else '\u2705 LOW'
            })
        print(pd.DataFrame(summary_data).to_string(index=False))
        print("\n\u2705 Analysis Complete! Green = Robust, Red = Fragile")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# UNIVERSAL STRATEGY EXPORT — Data Files + PDF Tearsheet
# ════════════════════════════════════════════════════════════════════
# INSTRUCTIONS:
#   1. Paste at the END of any strategy notebook
#   2. Edit STRATEGY_NAME and PARAM_COLS below
#   3. Run — exports structured data + a comprehensive PDF report
# ════════════════════════════════════════════════════════════════════

import os, sys, json, datetime, hashlib, platform
from matplotlib.backends.backend_pdf import PdfPages

# ═══ EDIT THESE LINES ═══════════════════════════════════════
STRATEGY_NAME = "ATR_Volatility_Breakout"                              # ← EDIT
PARAM_COLS    = ["atr_period", "sma_period", "atr_mult"]              # ← EDIT
NOTES         = ""                                                     # ← Optional run notes
# ════════════════════════════════════════════════════════════════

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'D'

# ── Google Drive mount ──
EXPORT_DIR = "/content/strategy_exports"
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
    IN_COLAB = True
    print("\u2705 Google Drive mounted")
except:
    print("\u26a0\ufe0f Local mode \u2014 exports to ./strategy_exports")

RUN_TIMESTAMP = datetime.datetime.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S")

# ── Folder structure ──
STRAT_DIR   = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
LATEST_DIR  = os.path.join(STRAT_DIR, "latest")
ARCHIVE_DIR = os.path.join(STRAT_DIR, "archive")
os.makedirs(LATEST_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
# Signal function — ATR Volatility Breakout
# ════════════════════════════════════════════════════════════════
def _compute_signals(price_s, params, high_s=None, low_s=None):
    idx = price_s.index; vals = price_s.values.astype(float)
    h_v = high_s.values.astype(float) if high_s is not None else vals
    l_v = low_s.values.astype(float) if low_s is not None else vals
    atr = talib.ATR(h_v, l_v, vals, timeperiod=params['atr_period'])
    sma = talib.SMA(vals, timeperiod=params['sma_period'])
    upper_band = sma + (params['atr_mult'] * atr)
    lower_band = sma - (params['atr_mult'] * atr)
    ub = pd.Series(upper_band, index=idx).shift(1)
    lb = pd.Series(lower_band, index=idx).shift(1)
    cs = pd.Series(vals, index=idx)
    e_raw = (cs > ub)
    x_raw = (cs < lb)
    entries = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
    exits = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
    return entries, exits

# ════════════════════════════════════════════════════════════════
# Build portfolios
# ════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
best_params = {}
for col in PARAM_COLS:
    if col == 'atr_mult':
        best_params[col] = float(best[col])
    else:
        best_params[col] = int(best[col])
param_str = ", ".join([f"{k}={v}" for k, v in best_params.items()])

if isinstance(stock_data.columns, pd.MultiIndex):
    full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
else:
    full_close = stock_data['Close'].astype(float).squeeze()
full_close.name = 'price'

split_idx = int(len(full_close) * 0.60)
train_close = full_close.iloc[:split_idx].copy()
val_close = full_close.iloc[split_idx:].copy()

# High/Low for ATR
if isinstance(stock_data.columns, pd.MultiIndex):
    high_s = stock_data[('High', TICKER)].astype(float).squeeze()
    low_s = stock_data[('Low', TICKER)].astype(float).squeeze()
else:
    high_s = stock_data['High'].astype(float).squeeze()
    low_s = stock_data['Low'].astype(float).squeeze()

# Full sample
e_full, x_full = _compute_signals(full_close, best_params, high_s, low_s)
pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                      init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# IS
e_is, x_is = _compute_signals(train_close, best_params,
                                high_s.iloc[:split_idx], low_s.iloc[:split_idx])
pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# OOS
e_oos, x_oos = _compute_signals(val_close, best_params,
                                  high_s.iloc[split_idx:], low_s.iloc[split_idx:])
pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                     init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# Buy & Hold
bh_e = pd.Series(False, index=full_close.index, dtype=bool); bh_e.iloc[0] = True
bh_x = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(close=full_close, entries=bh_e, exits=bh_x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# ── Extract metrics ──
trades_obj = pf_full.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
pos, neg = tr[tr > 0], tr[tr < 0]
years_full = max((full_close.index[-1] - full_close.index[0]).days / 365.25, 1e-9)
daily_rets = pf_full.returns()

def safe(fn, default=None):
    try: return float(fn())
    except: return default

M = {  # Master metrics dict
    'total_return': safe(pf_full.total_return), 'ann_return': safe(lambda: pf_full.annualized_return(freq=FREQ)),
    'sharpe': safe(lambda: pf_full.sharpe_ratio(freq=FREQ)), 'sortino': safe(lambda: pf_full.sortino_ratio(freq=FREQ)),
    'max_dd': safe(pf_full.max_drawdown), 'volatility': safe(lambda: pf_full.annualized_volatility(freq=FREQ)),
    'calmar': safe(lambda: pf_full.annualized_return(freq=FREQ)) / abs(safe(pf_full.max_drawdown)) if abs(safe(pf_full.max_drawdown, 0)) > 1e-9 else None,
    'trades': len(trades_obj), 'trades_yr': len(trades_obj) / years_full,
    'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
    'pf': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
    'expectancy': float(tr.mean()) if len(tr) > 0 else None,
    'avg_win': float(pos.mean()) if len(pos) > 0 else None, 'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
    'largest_win': float(pos.max()) if len(pos) > 0 else None, 'largest_loss': float(neg.min()) if len(neg) > 0 else None,
    'payoff': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
    'is_sharpe': safe(lambda: pf_is.sharpe_ratio(freq=FREQ)), 'is_return': safe(pf_is.total_return),
    'is_dd': safe(pf_is.max_drawdown), 'is_trades': len(pf_is.trades),
    'oos_sharpe': safe(lambda: pf_oos.sharpe_ratio(freq=FREQ)), 'oos_return': safe(pf_oos.total_return),
    'oos_dd': safe(pf_oos.max_drawdown), 'oos_trades': len(pf_oos.trades),
    'bh_return': safe(pf_bh.total_return), 'bh_sharpe': safe(lambda: pf_bh.sharpe_ratio(freq=FREQ)),
    'bh_dd': safe(pf_bh.max_drawdown),
}

# ════════════════════════════════════════════════════════════════
# 1. SAVE STRUCTURED DATA FILES
# ════════════════════════════════════════════════════════════════
export_json = {
    "metadata": {
        "run_id": RUN_ID, "export_timestamp": RUN_TIMESTAMP.isoformat(),
        "export_date_human": RUN_TIMESTAMP.strftime("%B %d, %Y at %I:%M %p"),
        "strategy_name": STRATEGY_NAME, "strategy_family": STRATEGY_NAME.split("_")[0],
        "ticker": TICKER,
        "instrument_type": ("crypto" if "-USD" in TICKER and TICKER.replace("-USD","").isalpha()
                           else "forex" if "/" in TICKER or (len(TICKER) == 6 and TICKER.isalpha())
                           else "equity/etf"),
        "data_source": "yfinance", "data_interval": "1d", "currency": "USD",
        "start_date": str(full_close.index[0].date()), "end_date": str(full_close.index[-1].date()),
        "total_bars": len(full_close), "total_years": round(years_full, 2),
        "train_start": str(train_close.index[0].date()), "train_end": str(train_close.index[-1].date()),
        "train_bars": len(train_close), "val_start": str(val_close.index[0].date()),
        "val_end": str(val_close.index[-1].date()), "val_bars": len(val_close), "train_ratio": 0.60,
        "init_cash": INIT_CASH, "fees_pct": FEES, "slippage_pct": SLIPPAGE, "frequency": FREQ,
        "first_close": round(float(full_close.iloc[0]), 4), "last_close": round(float(full_close.iloc[-1]), 4),
        "python_version": sys.version.split()[0], "environment": "colab_pro" if IN_COLAB else "local",
        "grid_combos_tested": len(results_df), "param_columns": PARAM_COLS, "notes": NOTES,
    },
    "best_params": best_params,
    "metrics_full_sample": {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
    "metrics_in_sample": {k.replace('is_',''): v for k, v in M.items() if k.startswith('is_')},
    "metrics_out_of_sample": {k.replace('oos_',''): v for k, v in M.items() if k.startswith('oos_')},
    "metrics_buy_hold": {k.replace('bh_',''): v for k, v in M.items() if k.startswith('bh_')},
    "grid_search_summary": {
        "top5": results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio','total_return','max_drawdown']].to_dict('records'),
    }
}

# Save JSON
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
with open(os.path.join(ARCHIVE_DIR, f"{RUN_ID}_summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
print(f"\u2705 summary.json")

# Save CSVs
pd.DataFrame({'date': full_close.index.strftime('%Y-%m-%d'), 'strategy_return': daily_rets.values,
              'close': full_close.values, 'portfolio_value': pf_full.value().values
}).to_csv(os.path.join(LATEST_DIR, "daily_returns.csv"), index=False)
print(f"\u2705 daily_returns.csv")

pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl,
              'cumulative_pnl': np.cumsum(pnl), 'is_winner': tr > 0
}).to_csv(os.path.join(LATEST_DIR, "trades.csv"), index=False)
print(f"\u2705 trades.csv")

results_df.to_csv(os.path.join(LATEST_DIR, "grid_results.csv"), index=False)
print(f"\u2705 grid_results.csv")

# Run log
log_path = os.path.join(EXPORT_DIR, "run_log.csv")
log_entry = pd.DataFrame([{
    "run_id": RUN_ID, "timestamp": RUN_TIMESTAMP.isoformat(), "strategy": STRATEGY_NAME,
    "ticker": TICKER, "best_params": str(best_params),
    "sharpe_full": round(M['sharpe'] or 0, 4), "sharpe_is": round(M['is_sharpe'] or 0, 4),
    "sharpe_oos": round(M['oos_sharpe'] or 0, 4), "total_return": round(M['total_return'] or 0, 4),
    "max_drawdown": round(M['max_dd'] or 0, 4), "total_trades": M['trades'],
    "win_rate": round(M['win_rate'] or 0, 1), "profit_factor": round(M['pf'] or 0, 2) if M['pf'] else None,
    "notes": NOTES, "export_path": STRAT_DIR,
}])
if os.path.exists(log_path):
    log_combined = pd.concat([pd.read_csv(log_path), log_entry], ignore_index=True)
else:
    log_combined = log_entry
log_combined.to_csv(log_path, index=False)
print(f"\u2705 run_log.csv ({len(log_combined)} runs)")

# ════════════════════════════════════════════════════════════════
# 2. GENERATE PDF TEARSHEET — Clean White Card Design
# ════════════════════════════════════════════════════════════════
pdf_path = os.path.join(LATEST_DIR, "tearsheet.pdf")
pdf_archive = os.path.join(ARCHIVE_DIR, f"{RUN_ID}_tearsheet.pdf")

fmt = lambda v, d=2: f"{v:.{d}f}" if v is not None and not np.isnan(v) else "N/A"
fmtp = lambda v: f"{v:.2%}" if v is not None and not np.isnan(v) else "N/A"

# ── Design tokens ──
BG       = '#FFFFFF'
CARD_BG  = '#F7F8FA'
CARD_BRD = '#E2E5EB'
TEXT_PRI = '#1A1D23'
TEXT_SEC = '#5A6270'
TEXT_MUT = '#8C95A3'
ACCENT   = '#2563EB'   # Blue
GREEN    = '#059669'
RED      = '#DC2626'
ORANGE   = '#EA580C'
GRID_CLR = '#E5E7EB'

def _draw_card(ax_fig, x, y, w, h, label, value, color=ACCENT, fontsize_val=22):
    """Draw a rounded metric card on the figure."""
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                           facecolor=CARD_BG, edgecolor=CARD_BRD, linewidth=1.2,
                           transform=ax_fig.transAxes, zorder=2)
    ax_fig.add_patch(rect)
    ax_fig.text(x + w/2, y + h*0.68, value, ha='center', va='center',
                fontsize=fontsize_val, fontweight='bold', color=color,
                transform=ax_fig.transAxes, zorder=3)
    ax_fig.text(x + w/2, y + h*0.25, label, ha='center', va='center',
                fontsize=8, color=TEXT_SEC, transform=ax_fig.transAxes, zorder=3)

def _style_ax(ax, title=None):
    """Apply clean white styling to an axis."""
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT_SEC, labelsize=8)
    ax.grid(True, alpha=0.4, color=GRID_CLR, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_color(CARD_BRD)
        spine.set_linewidth(0.8)
    if title:
        ax.set_title(title, color=TEXT_PRI, fontsize=11, fontweight='bold', pad=10)

with PdfPages(pdf_path) as pdf:

    # ── PAGE 1: Dashboard with Metric Cards + Summary Table ──
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    # Header bar
    from matplotlib.patches import FancyBboxPatch, Rectangle
    header = Rectangle((0, 0.91), 1, 0.09, facecolor=ACCENT, transform=ax.transAxes, zorder=1)
    ax.add_patch(header)
    ax.text(0.04, 0.955, "STRATEGY TEARSHEET", ha='left', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes, zorder=2)
    ax.text(0.96, 0.955, f"{STRATEGY_NAME}  |  {TICKER}", ha='right', va='center',
            fontsize=12, color='rgba(255,255,255,0.85)', transform=ax.transAxes, zorder=2, family='monospace')
    ax.text(0.96, 0.92, f"{full_close.index[0].date()} to {full_close.index[-1].date()}  |  {len(full_close)} bars  |  {param_str}",
            ha='right', va='center', fontsize=8, color='rgba(255,255,255,0.65)', transform=ax.transAxes, zorder=2)

    # Metric cards row — top KPIs
    card_w, card_h = 0.145, 0.09
    card_y = 0.79
    cards = [
        ("SHARPE", fmt(M['sharpe'], 3), ACCENT),
        ("RETURN", fmtp(M['total_return']), GREEN if (M['total_return'] or 0) > 0 else RED),
        ("MAX DD", fmtp(M['max_dd']), RED if (M['max_dd'] or 0) < -0.15 else ORANGE),
        ("WIN RATE", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", GREEN if (M['win_rate'] or 0) > 50 else RED),
        ("TRADES", str(M['trades']), TEXT_PRI),
        ("PROFIT FACTOR", fmt(M['pf']), GREEN if (M['pf'] or 0) > 1.5 else (ORANGE if (M['pf'] or 0) > 1 else RED)),
    ]
    x_start = 0.03
    gap = (0.94 - len(cards) * card_w) / (len(cards) - 1)
    for i, (label, value, color) in enumerate(cards):
        cx = x_start + i * (card_w + gap)
        _draw_card(ax, cx, card_y, card_w, card_h, label, value, color)

    # IS vs OOS comparison table
    table_y = 0.04
    table_h = 0.70
    rows = [
        ["METRIC", "FULL SAMPLE", "IN-SAMPLE", "OUT-OF-SAMPLE", "BUY & HOLD"],
        ["Total Return", fmtp(M['total_return']), fmtp(M['is_return']), fmtp(M['oos_return']), fmtp(M['bh_return'])],
        ["Sharpe Ratio", fmt(M['sharpe'], 3), fmt(M['is_sharpe'], 3), fmt(M['oos_sharpe'], 3), fmt(M['bh_sharpe'], 3)],
        ["Sortino Ratio", fmt(M['sortino'], 3), "—", "—", "—"],
        ["Max Drawdown", fmtp(M['max_dd']), fmtp(M['is_dd']), fmtp(M['oos_dd']), fmtp(M['bh_dd'])],
        ["Calmar Ratio", fmt(M['calmar'], 3), "—", "—", "—"],
        ["Volatility (Ann.)", fmtp(M['volatility']), "—", "—", "—"],
        ["Win Rate", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", "—", "—", "—"],
        ["Profit Factor", fmt(M['pf']), "—", "—", "—"],
        ["Expectancy", fmt(M['expectancy'], 4), "—", "—", "—"],
        ["Payoff Ratio", fmt(M['payoff']), "—", "—", "—"],
        ["Total Trades", str(M['trades']), str(M['is_trades']), str(M['oos_trades']), "1"],
        ["Trades/Year", fmt(M['trades_yr'], 1), "—", "—", "—"],
        ["Avg Win", fmtp(M['avg_win']), "—", "—", "—"],
        ["Avg Loss", fmtp(M['avg_loss']), "—", "—", "—"],
        ["Largest Win", fmtp(M['largest_win']), "—", "—", "—"],
        ["Largest Loss", fmtp(M['largest_loss']), "—", "—", "—"],
    ]

    table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
                     bbox=[0.03, table_y, 0.94, table_h])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(CARD_BRD)
        cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor(ACCENT)
            cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor(BG if row % 2 == 1 else CARD_BG)
            cell.set_text_props(color=TEXT_PRI, fontsize=8, family='monospace')
            if col == 0:
                cell.set_text_props(color=TEXT_PRI, fontsize=8, fontweight='bold')
        cell.set_height(0.052)

    # Footer
    ax.text(0.5, 0.01, f"Run {RUN_ID}  |  QS Finance", ha='center', va='bottom',
            fontsize=7, color=TEXT_MUT, transform=ax.transAxes)

    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

    # ── PAGE 2: Equity Curve + Drawdown ──
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor(BG)
    fig.suptitle(f'{STRATEGY_NAME} on {TICKER} — Equity & Drawdown', fontsize=14,
                 fontweight='bold', color=TEXT_PRI, y=0.97)

    eq_strat = pf_full.value(); eq_bh = pf_bh.value()

    # Equity curve with gradient fill
    _style_ax(ax1)
    ax1.plot(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values,
             color=ACCENT, linewidth=2, label='Strategy (IS)', solid_capstyle='round')
    ax1.plot(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values,
             color=ORANGE, linewidth=2, label='Strategy (OOS)', solid_capstyle='round')
    ax1.plot(full_close.index, eq_bh.values, color=TEXT_MUT, linewidth=1.2,
             alpha=0.6, linestyle='--', label='Buy & Hold')
    ax1.axvline(x=full_close.index[split_idx], color=RED, linestyle=':', alpha=0.3, linewidth=1)
    ax1.fill_between(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values,
                      eq_strat.iloc[:split_idx].values.min(), alpha=0.06, color=ACCENT)
    ax1.fill_between(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values,
                      eq_strat.iloc[split_idx:].values.min(), alpha=0.06, color=ORANGE)
    ax1.set_ylabel('Portfolio Value ($)', color=TEXT_SEC, fontsize=9)
    ax1.legend(fontsize=8, facecolor=BG, edgecolor=CARD_BRD, labelcolor=TEXT_PRI, framealpha=0.95)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Stats banner
    stats_text = f"Sharpe: {fmt(M['sharpe'],3)}   |   Return: {fmtp(M['total_return'])}   |   MaxDD: {fmtp(M['max_dd'])}   |   WR: {M['win_rate']:.1f}%   |   PF: {fmt(M['pf'])}"
    ax1.text(0.5, 0.03, stats_text, transform=ax1.transAxes, fontsize=8, ha='center',
             color=TEXT_SEC, family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))

    # Drawdown
    _style_ax(ax2)
    dd = pf_full.drawdown() * 100
    ax2.fill_between(full_close.index, dd.values, 0, color=RED, alpha=0.2)
    ax2.plot(full_close.index, dd.values, color=RED, linewidth=0.8, alpha=0.7)
    ax2.set_ylabel('Drawdown %', color=TEXT_SEC, fontsize=9)
    ax2.set_xlabel('Date', color=TEXT_SEC, fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

    # ── PAGE 3: Trade Analysis — 2x2 Grid ──
    if len(tr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
        fig.patch.set_facecolor(BG)
        fig.suptitle(f'Trade-by-Trade Analysis — {len(tr)} Trades', fontsize=14,
                     fontweight='bold', color=TEXT_PRI, y=0.97)

        n = len(tr)
        colors_bar = [GREEN if r > 0 else RED for r in tr]
        colors_pnl = [GREEN if p > 0 else RED for p in pnl]

        for a in axes.flat:
            _style_ax(a)

        # Trade returns bar chart
        axes[0,0].bar(range(n), tr*100, color=colors_bar, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,0].axhline(np.mean(tr)*100, color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: {np.mean(tr)*100:.2f}%')
        axes[0,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,0].set_title('Trade Returns (%)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,0].set_xlabel('Trade #', color=TEXT_SEC, fontsize=8)
        axes[0,0].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        # Trade P&L bar chart
        axes[0,1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,1].axhline(np.mean(pnl), color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: ${np.mean(pnl):,.0f}')
        axes[0,1].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,1].set_title('Trade P&L ($)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        # Cumulative P&L with gradient
        cum_pnl = np.cumsum(pnl)
        axes[1,0].plot(range(1, n+1), cum_pnl, color=ACCENT, linewidth=2.5, solid_capstyle='round')
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl>=0, alpha=0.12, color=GREEN)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl<0, alpha=0.12, color=RED)
        axes[1,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[1,0].set_title('Cumulative P&L', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,0].set_xlabel('Trade #', color=TEXT_SEC, fontsize=8)
        axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        # Return distribution histogram
        axes[1,1].hist(tr*100, bins=min(30, max(10, n//3)), color=ACCENT, edgecolor='white',
                       alpha=0.75, linewidth=0.5)
        axes[1,1].axvline(np.mean(tr)*100, color=RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[1,1].axvline(0, color=TEXT_MUT, linewidth=0.8, alpha=0.5)
        axes[1,1].set_title('Return Distribution', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,1].set_xlabel('Return %', color=TEXT_SEC, fontsize=8)
        axes[1,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        pdf.savefig(fig, facecolor=BG)
        plt.close(fig)

    # ── PAGE 4: Monte Carlo FTMO Simulation ──
    dr = daily_rets.values.ravel(); dr = dr[~np.isnan(dr)]
    N_SIM = 5000; DAYS = 30; ACCOUNT = 100000
    np.random.seed(42)
    n_passed = n_blown_t = n_blown_d = 0
    final_eqs = np.zeros(N_SIM); sample_paths = []

    for s in range(N_SIM):
        eq = ACCOUNT; path = [ACCOUNT]
        sim_rets = np.random.choice(dr, size=DAYS, replace=True)
        blown = False
        for d in range(DAYS):
            day_start = eq; eq *= (1 + sim_rets[d]); path.append(eq)
            if (eq - day_start)/ACCOUNT < -0.05: n_blown_d += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT < -0.10: n_blown_t += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT >= 0.10: n_passed += 1; blown = True; break
        while len(path) < DAYS + 1: path.append(path[-1])
        final_eqs[s] = path[-1]
        if s < 150: sample_paths.append(path)

    n_still = N_SIM - n_passed - n_blown_t - n_blown_d
    pass_rate = n_passed / N_SIM * 100

    verdict = "FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25 else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY"
    verdict_color = GREEN if pass_rate >= 50 else ORANGE if pass_rate >= 25 else (ORANGE if pass_rate >= 10 else RED)

    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)

    # Top section: MC result cards
    ax_top = fig.add_axes([0, 0.82, 1, 0.18])
    ax_top.set_xlim(0, 1); ax_top.set_ylim(0, 1); ax_top.axis('off')

    # Title
    ax_top.text(0.5, 0.85, f'FTMO Monte Carlo — {N_SIM:,} Simulations', ha='center', va='center',
                fontsize=16, fontweight='bold', color=TEXT_PRI, transform=ax_top.transAxes)

    # Result cards
    mc_cards = [
        ("PASS RATE", f"{pass_rate:.1f}%", verdict_color),
        ("VERDICT", verdict, verdict_color),
        ("PASSED", f"{n_passed:,}", GREEN),
        ("BLOWN (TOTAL)", f"{n_blown_t:,}", RED),
        ("BLOWN (DAILY)", f"{n_blown_d:,}", RED),
        ("STILL TRADING", f"{n_still:,}", TEXT_SEC),
    ]
    mc_cw = 0.14
    mc_gap = (0.92 - len(mc_cards) * mc_cw) / (len(mc_cards) - 1)
    for i, (label, value, color) in enumerate(mc_cards):
        cx = 0.04 + i * (mc_cw + mc_gap)
        _draw_card(ax_top, cx, 0.05, mc_cw, 0.65, label, value, color, fontsize_val=16)

    # Bottom section: Equity paths
    ax_mc = fig.add_axes([0.08, 0.08, 0.86, 0.70])
    _style_ax(ax_mc)

    for path in sample_paths:
        c = GREEN if path[-1] >= 110000 else (RED if path[-1] <= 90000 else TEXT_MUT)
        ax_mc.plot(range(DAYS+1), path, color=c, alpha=0.15, linewidth=0.5)
    ax_mc.axhline(110000, color=GREEN, linestyle='--', linewidth=2.5, label=f'10% Target ($110k)')
    ax_mc.axhline(90000, color=RED, linestyle='--', linewidth=2.5, label=f'10% Max Loss ($90k)')
    ax_mc.axhline(100000, color=TEXT_MUT, linestyle='-', linewidth=0.8, alpha=0.4)

    ax_mc.set_xlabel('Trading Day', color=TEXT_SEC, fontsize=9)
    ax_mc.set_ylabel('Equity ($)', color=TEXT_SEC, fontsize=9)
    ax_mc.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax_mc.legend(fontsize=9, facecolor=BG, edgecolor=CARD_BRD, labelcolor=TEXT_PRI, framealpha=0.95)

    # Median final equity annotation
    ax_mc.text(0.5, 0.03, f"Median Final Equity: ${np.median(final_eqs):,.0f}", transform=ax_mc.transAxes,
               fontsize=9, ha='center', color=TEXT_SEC, family='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))

    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

# Copy PDF to archive
import shutil
shutil.copy2(pdf_path, pdf_archive)

# Add MC results to JSON
mc_data = {"pass_rate": round(pass_rate, 1), "n_simulations": N_SIM, "n_passed": n_passed,
           "n_blown_total": n_blown_t, "n_blown_daily": n_blown_d, "n_still_trading": n_still,
           "median_final_equity": round(float(np.median(final_eqs)), 2), "verdict": verdict}
export_json["monte_carlo_ftmo"] = mc_data
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)


# ════════════════════════════════════════════════════════════════
# LOG TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════════
try:
    from lib.sheets_logger import SheetsLogger
    _logger = SheetsLogger()
    _logger.log_result(export_json)
    # Log trades if available
    _trades_path = os.path.join(LATEST_DIR, "trades.csv")
    if os.path.exists(_trades_path):
        _trades_df = pd.read_csv(_trades_path)
        _logger.log_trades(TICKER, STRATEGY_NAME, RUN_ID, _trades_df)
except Exception as _e:
    print(f"⚠️ Sheets logging skipped: {_e}")

# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"\U0001f4e6 EXPORT COMPLETE \u2014 {STRATEGY_NAME} on {TICKER}")
print(f"{'='*70}")
print(f"  Run ID:       {RUN_ID}")
print(f"  Timestamp:    {RUN_TIMESTAMP.strftime('%B %d, %Y at %I:%M:%S %p')}")
print(f"  Instrument:   {TICKER} ({export_json['metadata']['instrument_type']})")
print(f"  Params:       {param_str}")
print(f"  Sharpe:       {fmt(M['sharpe'],3)} (IS: {fmt(M['is_sharpe'],3)} -> OOS: {fmt(M['oos_sharpe'],3)})")
print(f"  FTMO Verdict: {verdict} ({pass_rate:.1f}% pass rate)")
print(f"{'='*70}")
print(f"\n\U0001f4c2 {STRAT_DIR}/")
print(f"  \u251c\u2500\u2500 latest/")
print(f"  \u2502   \u251c\u2500\u2500 summary.json")
print(f"  \u2502   \u251c\u2500\u2500 daily_returns.csv")
print(f"  \u2502   \u251c\u2500\u2500 trades.csv")
print(f"  \u2502   \u251c\u2500\u2500 grid_results.csv")
print(f"  \u2502   \u2514\u2500\u2500 tearsheet.pdf       \u2190 Share this with Claude!")
print(f"  \u2514\u2500\u2500 archive/")
print(f"      \u251c\u2500\u2500 {RUN_ID}_summary.json")
print(f"      \u2514\u2500\u2500 {RUN_ID}_tearsheet.pdf")
print(f"\n\U0001f4cb run_log.csv ({len(log_combined)} total runs)")
print(f"\n\U0001f4a1 To get my analysis: upload the tearsheet.pdf to our chat.")
print(f"   For deeper analysis: also upload summary.json + daily_returns.csv")